# Week 5 Practical: Sentiment Analysis with Logistic Regression

This lab follows the Week 5 lecture sequence:

**sentiment analysis → vocabulary → sparse representations → preprocessing → positive/negative frequency counts → 3-feature extraction → feature matrix → logistic regression → training → testing → accuracy**

We use the lecture notation throughout:

$$
f(X,\mathbf w),\qquad \mathbf w,\qquad J(\mathbf w),\qquad pred
$$

Dataset: `week5_sentiment_tweets.csv`

- `label = 1`: positive sentiment
- `label = 0`: negative sentiment

> The dataset is synthetic and for machine-learning education only.

## Setup

In [ ]:
import re
import string
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from nltk.stem import PorterStemmer
    stemmer = PorterStemmer()
except ImportError:
    stemmer = None

DATA_FILE = "week5_sentiment_tweets.csv"
np.set_printoptions(suppress=True, precision=4)

# 1. Sentiment Analysis


Sentiment analysis is a binary classification task with $y\in\{0,1\}$.

### TODO 1
1. Load the CSV into `df`.
2. Show the first 10 rows.
3. Print shape, dtypes, and missing-value counts.
4. Count positive and negative examples.
5. Print three examples from each class.
6. Explain why text must be converted to numerical features before logistic regression can use it.

In [ ]:
# TODO 1
# df = pd.read_csv(DATA_FILE)
# ...

# 2. Vocabulary


A vocabulary is the set of unique tokens occurring in the corpus.

### TODO 2
1. Lowercase each tweet.
2. Split on whitespace.
3. Build the set of unique tokens.
4. Print vocabulary size and 30 example tokens.
5. Identify noisy entries caused by punctuation, URLs, mentions, or casing.
6. Explain why preprocessing should happen before building a final vocabulary.

In [ ]:
# TODO 2
# raw_vocab = ...

# 3. Sparse Representations


A bag-of-words vector can have one dimension per vocabulary item. Most entries are usually zero.

### TODO 3
1. Select one tweet.
2. Create a binary bag-of-words vector over the vocabulary from TODO 2.
3. Report vector length, number of nonzero entries, and percentage of zeros.
4. Explain why large vocabularies increase memory, training time, and prediction time.
5. Explain why the 3-feature method used later is much smaller.

In [ ]:
# TODO 3
# selected_tweet = ...
# bow_vector = ...

# 4. Text Preprocessing


We will lowercase, remove URLs and mentions, remove punctuation, remove selected stop words, and stem words.

### TODO 4
Implement:

```python
process_tweet(text)
```

Your function must:
1. lowercase,
2. remove URLs,
3. remove `@username` mentions,
4. remove punctuation,
5. tokenize,
6. remove stop words,
7. stem remaining tokens,
8. return a list of processed tokens.

In [ ]:
STOP_WORDS = {
    "i","me","my","we","our","you","your","he","she","it","they","them",
    "am","is","are","was","were","be","been","a","an","the","and","or","but",
    "to","of","for","in","on","at","this","that","these","those"
}

def process_tweet(text):
    raise NotImplementedError("Complete TODO 4")

# 5. Stop Words and Punctuation


Stop-word removal is task dependent. For sentiment analysis, words such as **not** can be important.

### TODO 5
1. Process five noisy tweets and show before/after forms.
2. Compare raw vocabulary size with processed vocabulary size.
3. Temporarily remove `not` and inspect negative tweets containing it.
4. Explain why deleting `not` can reverse or weaken sentiment information.
5. Give one other task where a common word might still be important.

In [ ]:
# TODO 5
# Compare original and processed tweets

# 6. Stemming and Lowercasing


Lowercasing merges forms such as GREAT/Great/great. Stemming reduces related forms such as tuning/tuned/tunes.

### TODO 6
1. Process: `TUNING`, `tuned`, `tunes`, `learning`, `learned`, `HAPPY`.
2. Display original and processed forms.
3. Identify stems that are not ordinary English words.
4. Explain stemming vs. lemmatization.
5. Explain why stemming can still help classification.

In [ ]:
# TODO 6
test_words = ["TUNING","tuned","tunes","learning","learned","HAPPY"]
# ...

# 7. Positive and Negative Frequency Counts


For each processed word \(w\), store:

\[
freqs(w,1),\qquad freqs(w,0)
\]

### TODO 7
Implement:

```python
build_freqs(texts, labels)
```

1. Process every tweet.
2. Increment `(word,1)` for positive tweets.
3. Increment `(word,0)` for negative tweets.
4. Inspect counts for several words.
5. Identify strongly positive, strongly negative, and neutral words.
6. Explain what this dictionary captures beyond a simple vocabulary.

In [ ]:
def build_freqs(texts, labels):
    raise NotImplementedError("Complete TODO 7")

# 8. Visualizing Word Frequencies


Each word can be represented by:

$$
(freqs(w,1), freqs(w,0))
$$

### TODO 8
1. Choose at least 10 words.
2. Plot positive frequency on x-axis and negative frequency on y-axis.
3. Annotate each point with its word.
4. Explain where positive, negative, and neutral words appear.
5. Identify one surprising association.

In [ ]:
# TODO 8
# Create scatter plot

# 9. Three-Dimensional Feature Extraction


Each tweet becomes:

$$
\mathbf{x}^{(i)}
=
[1,\text{sum positive frequencies},\text{sum negative frequencies}]
$$

### TODO 9
Implement:

```python
extract_features(text, freqs)
```

1. Preprocess the tweet.
2. Start with bias = 1.
3. Sum positive frequencies for every word.
4. Sum negative frequencies for every word.
5. Return `np.array([1, pos_sum, neg_sum], dtype=float)`.
6. Test on one positive, one negative, and one ambiguous tweet.
7. Interpret all three values.

In [ ]:
def extract_features(text, freqs):
    raise NotImplementedError("Complete TODO 9")

# 10. Constructing the Feature Matrix


For $m$ tweets:

$$
X\in\mathbb{R}^{m\times3}
$$

### TODO 10
1. Initialize `X = np.zeros((m,3))`.
2. Fill one row per tweet using `extract_features`.
3. Build `y`.
4. Print shapes and first five rows.
5. Verify column 0 is all ones.
6. Explain why there are only 3 columns regardless of vocabulary size.

In [ ]:
# TODO 10
# X = np.zeros((m, 3))
# ...

# 11. Train/Test Split Without Leakage


The frequency dictionary must be built from **training data only**.

### TODO 11
1. Shuffle with a fixed random seed.
2. Create 80% train / 20% test split.
3. Build `freqs_train` from training tweets only.
4. Extract both train and test features using `freqs_train`.
5. Print all shapes.
6. Explain why using test labels to build frequencies is data leakage.

In [ ]:
# TODO 11
rng = np.random.default_rng(42)
# ...

# 12. Logistic Regression Model


For the 3-feature vector and weights:

$$
\mathbf w=[w_0,w_1,w_2]^T
$$

$$
f(X,\mathbf w)=\sigma(X\mathbf w)
$$

where:

$$
\sigma(z)=\frac{1}{1+e^{-z}}
$$

### TODO 12
1. Implement `sigmoid(z)`.
2. Implement `f(X,w)`.
3. Test with zero weights.
4. Explain why zero weights produce probability 0.5 for every example.

In [ ]:
def sigmoid(z):
    raise NotImplementedError("Complete TODO 12")

def f(X, w):
    raise NotImplementedError("Complete TODO 12")

# 13. Logistic Regression Cost


$$
J(\mathbf w)=
-\frac{1}{m}\sum_i
[y^{(i)}\log f^{(i)}+(1-y^{(i)})\log(1-f^{(i)})]
$$

### TODO 13
Implement `compute_cost(X,y,w)`.

1. Compute probabilities with `f`.
2. Clip probabilities away from 0 and 1.
3. Compute average binary cross-entropy.
4. Evaluate cost at zero weights and another manual weight vector.
5. Explain what a lower cost means.

In [ ]:
def compute_cost(X, y, w):
    raise NotImplementedError("Complete TODO 13")

# 14. Gradient


The vectorized gradient is:

$$
\nabla J(\mathbf w)
=
\frac{1}{m}X^T(f(X,\mathbf w)-y)
$$

### TODO 14
Implement `compute_gradient(X,y,w)`.

1. Compute predictions.
2. Compute residuals.
3. Use matrix multiplication.
4. Verify gradient shape equals weight shape.
5. Inspect the gradient at zero initialization.
6. Interpret the signs of the positive- and negative-frequency gradients.

In [ ]:
def compute_gradient(X, y, w):
    raise NotImplementedError("Complete TODO 14")

# 15. Training Logistic Regression


Gradient descent updates:

$$
\mathbf w\leftarrow\mathbf w-\alpha\nabla J(\mathbf w)
$$

### TODO 15
Implement `gradient_descent(X,y,w_init,alpha,num_iterations)`.

1. Update all weights simultaneously.
2. Record cost each iteration.
3. Return learned weights and cost history.
4. Train on training data.
5. Try at least three learning rates.
6. Plot cost vs. iteration.
7. Identify slow, good, and unstable learning rates.

In [ ]:
def gradient_descent(X, y, w_init, alpha, num_iterations):
    raise NotImplementedError("Complete TODO 15")

# 16. Interpreting Learned Weights


The model learns $w_0,w_1,w_2$ for bias, positive frequency, and negative frequency.

### TODO 16
1. Print learned weights.
2. State the signs of $w_1$ and $w_2$.
3. Explain why $w_1$ should usually be positive and $w_2$ negative.
4. Compute $\mathbf w^T\mathbf x$ for one positive and one negative test tweet.
5. Convert each score to a probability.
6. Interpret the results.

In [ ]:
# TODO 16
# ...

# 17. Testing Logistic Regression


For test data:

$$
f(X_{test},\mathbf w)
$$

and:

$$
pred=\mathbf{1}(f(X_{test},\mathbf w)\ge0.5)
$$

### TODO 17
1. Compute test probabilities.
2. Threshold at 0.5 to obtain `pred`.
3. Print first 15 probabilities, predictions, and true labels.
4. Find confident predictions and uncertain predictions near 0.5.
5. Explain the difference between probability 0.51 and 0.95.

In [ ]:
# TODO 17
# test_prob = f(X_test, w_learned)
# pred = ...

# 18. Accuracy


$$
Accuracy=
\frac{1}{m}\sum_i \mathbf{1}(pred^{(i)}=y^{(i)})
$$

### TODO 18
1. Compare `pred == y_test`.
2. Compute accuracy manually.
3. Count correct and incorrect predictions.
4. Print decimal and percentage accuracy.
5. Verify with `np.mean(pred == y_test)`.
6. Explain why accuracy may be insufficient for imbalanced data.

In [ ]:
# TODO 18
# comparison = ...
# accuracy = ...

# 19. Error Analysis


### TODO 19
1. Compute TP, TN, FP, FN manually.
2. Verify they sum to the number of test examples.
3. Print at least five misclassified tweets.
4. Show text, true label, probability, and prediction.
5. Discuss likely causes such as negation, ambiguity, unseen words, or preprocessing.

In [ ]:
# TODO 19
# TP = ...
# TN = ...
# FP = ...
# FN = ...

# 20. Threshold Sensitivity


For threshold $\tau$:

$$
pred_\tau=\mathbf{1}(f(X,\mathbf w)\ge\tau)
$$

### TODO 20
Compare thresholds 0.3, 0.5, and 0.7.

For each, compute:
- predicted positives,
- predicted negatives,
- accuracy,
- TP,
- TN,
- FP,
- FN.

Put the results in a DataFrame and explain the trade-off.

In [ ]:
# TODO 20
thresholds = [0.3, 0.5, 0.7]
# ...

# 21. Final End-to-End Challenge


### TODO 21
Starting from the raw CSV:

1. load the dataset,
2. split train/test,
3. preprocess tweets,
4. build the training frequency dictionary,
5. extract `[1, sum positive, sum negative]`,
6. create `X_train` and `X_test`,
7. initialize \(\mathbf w\),
8. train logistic regression from scratch,
9. plot cost history,
10. compute test probabilities,
11. threshold at 0.5,
12. compute accuracy and TP/TN/FP/FN,
13. inspect correct and incorrect predictions,
14. classify three new sentences you write.

**Constraint:** do not use `sklearn.linear_model.LogisticRegression`.

In [ ]:
# TODO 21
# Complete the full pipeline here

# 22. Concept Questions

### TODO 22

Answer each in 3–6 sentences.

1. Why can logistic regression classify text even though it operates on numbers?
2. Why do full bag-of-words vectors become sparse and expensive?
3. What information is lost when the vocabulary is compressed to only `[1, sum positive, sum negative]`?
4. Why can aggressive preprocessing hurt model performance?
5. Why should `not` often be preserved in sentiment analysis?
6. What is gained and lost through stemming?
7. Why must the frequency dictionary be built only from training data?
8. Why should the positive-frequency weight usually be positive and the negative-frequency weight negative?
9. What is the difference between probabilities 0.51 and 0.99 if both produce class 1?
10. Give a scenario where 95% accuracy could still represent poor performance.

# 23. Submission Checklist

- [ ] Loaded the dataset correctly
- [ ] Built a vocabulary
- [ ] Demonstrated sparsity
- [ ] Implemented preprocessing
- [ ] Investigated stop words and punctuation
- [ ] Applied lowercasing and stemming
- [ ] Built `(word,class) → frequency` counts
- [ ] Visualized positive/negative word frequencies
- [ ] Implemented `[1, sum positive, sum negative]`
- [ ] Created an $m\times3$ feature matrix
- [ ] Avoided train/test leakage
- [ ] Implemented sigmoid
- [ ] Implemented $f(X,\mathbf w)$
- [ ] Implemented $J(\mathbf w)$
- [ ] Implemented the gradient
- [ ] Implemented gradient descent
- [ ] Plotted training cost
- [ ] Interpreted learned weights
- [ ] Tested on unseen data
- [ ] Computed accuracy
- [ ] Computed TP, TN, FP, FN
- [ ] Performed error analysis
- [ ] Compared thresholds
- [ ] Completed the end-to-end challenge

## Week 5 Summary

$$
\text{Raw text}
\rightarrow
\text{Preprocessing}
\rightarrow
\text{Frequency dictionary}
\rightarrow
[1,\text{sum positive},\text{sum negative}]
\rightarrow
f(X,\mathbf w)
\rightarrow
\text{probability}
\rightarrow
pred
$$

Training learns $\mathbf w$ by minimizing $J(\mathbf w)$. Testing compares `pred` with unseen true labels.